# v3w · v3 rebuilt with weekly decisions

Plan: [docs/weekly_rebuild_note.md](../docs/weekly_rebuild_note.md), fixed and hashed before this notebook existed. Everything in [docs/v3_design_note.md](../docs/v3_design_note.md) is kept: treatments C0–C3, the calibration rule, the 0.30% hurdle and both default rules. Two things change:

- **The base model is v2w's frozen model** (`outputs/weekly/v2/frozen_config.json`), not v2's gradient boosting. It is not re-tuned. Because the model changed, the calibration check (§5.1) is re-run first. That check uses no trading results.
- **Every backtest decides once a week:** after the week's last close, the trade fills at the next open, and the position is held all week.

Order:
1. calibration (2013–2022, probabilities only);
2. selection among C1–C3 × {long, cash} on the 2013–2022 weekly walk-forward, by Sharpe over T-bills;
3. freeze;
4. 2023–24 test and the 2025–26 holdout (both seen periods; see the plan).

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from src import config, v2, v3, weekly
from src.data import load_snapshot
from src.dataset import build_dataset_v2
from src.evaluate import classification_metrics

OUT = config.OUTPUT_DIR / "weekly" / "v3"; OUT.mkdir(parents=True, exist_ok=True)
BASE = json.loads((config.OUTPUT_DIR / "weekly" / "v2" / "frozen_config.json").read_text())
MODEL, PARAMS = BASE["model"], BASE["params"]
print("base model (v2w, frozen):", MODEL, PARAMS)
raw = load_snapshot(); px = raw[["Open", "Close"]]; ds = build_dataset_v2(raw)
rf = v2.load_tbill_daily(raw.index)
DEV = list(range(2013, 2023))

base model (v2w, frozen): LR {'clf__C': 0.014618962793704957}


## 1 · Calibration check on the new base model (§5.1; no trading)

In [2]:
PROBA = {k: v3.walk_forward_proba(ds, k, DEV, PARAMS, MODEL) for k in v3.CANDIDATES}
YEARLY = {k: v3.yearly_calibration(PROBA[k], ds["label"]) for k in v3.CANDIDATES}
calib = pd.DataFrame({k: {"mean P(Sell)": y["pred_Sell"].mean(), "actual Sell share": y["actual_Sell"].mean(),
                          "years within ±3 pp": int((y["sell_gap_pp"].abs() <= 3).sum()),
                          "passes §5.1": v3.calibration_passes(y), "log_loss": y["log_loss"].mean()}
                      for k, y in YEARLY.items()}).T
calib

,mean P(Sell),actual Sell share,years within ±3 pp,passes §5.1,log_loss
C0,0.305449,0.207348,1,False,1.029746
C1,0.217292,0.207348,4,False,1.006616
C2,0.216446,0.207348,4,False,1.007296
C3,0.240633,0.207348,3,False,1.019379


## 2 · Development walk-forward 2013–2022, weekly decisions; selection

In [3]:
days = ds.loc["2013-01-01":"2022-12-31"].index
rows, ER = [], {}
for kind in v3.CANDIDATES:
    ER[kind] = v3.expected_return(PROBA[kind])
    for default in v2.DEFAULTS:
        held = v2.held_from_positions(weekly.weekly_positions_from_er(ER[kind], default), start=1 if default == "long" else 0)
        s = v2.summarize(v2.simulate(held, px.loc[days], rf.loc[days]), held, rf.loc[days], "")
        rows.append({"id": f"{kind}-{default[0].upper()}", "eligible": kind != "C0", **{k: s[k] for k in ["final_value", "sharpe_excess", "max_drawdown", "round_trips", "exposure", "total_fees"]}})
bh_h = v2.buy_and_hold_held(days)
bh = v2.summarize(v2.simulate(bh_h, px.loc[days], rf.loc[days]), bh_h, rf.loc[days], "Buy & Hold")
dev = pd.DataFrame(rows).set_index("id")
display(dev.sort_values("sharpe_excess", ascending=False))
print("Buy & Hold:", {k: bh[k] for k in ["final_value", "sharpe_excess", "max_drawdown"]})
elig = dev[dev["eligible"]].sort_values(["sharpe_excess", "round_trips"], ascending=[False, True])
PICK = elig.index[0]
KIND, DEFAULT = PICK.split("-")[0], "long" if PICK.endswith("L") else "cash"
FROZEN = {"version": "v3w", "candidate": PICK, "treatment": KIND, "default": DEFAULT, "model": MODEL, "params": PARAMS,
          "schedule": "weekly", "dev_sharpe_excess": float(elig.loc[PICK, "sharpe_excess"]), "dev_bh_sharpe_excess": bh["sharpe_excess"],
          "calibration_passed": bool(calib.loc[KIND, "passes §5.1"]),
          "plan_sha256": (config.ROOT / "docs" / "weekly_rebuild_note.sha256").read_text().split()[0], "frozen_before_test": True}
(OUT / "frozen_config.json").write_text(json.dumps(FROZEN, indent=2, default=str))
FROZEN

,eligible,final_value,sharpe_excess,max_drawdown,round_trips,exposure,total_fees
id,,,,,,,
C3-L,True,316843.58,0.708,-0.3372,1,1.000,99.90
C2-L,True,267578.27,0.617,-0.3372,4,0.983,1423.82
C1-L,True,260568.96,0.603,-0.3372,3,0.989,928.01
C0-C,False,126989.43,0.513,-0.0469,7,0.021,1567.59
C0-L,False,182086.75,0.467,-0.2686,28,0.680,9513.17
C3-C,True,141640.68,0.257,-0.3372,20,0.508,5330.54
C1-C,True,126741.97,0.185,-0.3345,20,0.628,5549.56
C2-C,True,125963.85,0.181,-0.3345,21,0.628,5770.54


Buy & Hold: {'final_value': 316843.58, 'sharpe_excess': 0.708, 'max_drawdown': -0.3372}


{'version': 'v3w',
 'candidate': 'C3-L',
 'treatment': 'C3',
 'default': 'long',
 'model': 'LR',
 'params': {'clf__C': 0.014618962793704957},
 'schedule': 'weekly',
 'dev_sharpe_excess': 0.708,
 'dev_bh_sharpe_excess': 0.708,
 'calibration_passed': False,
 'plan_sha256': '82602bdd44c1797b167933a11fe610c823d14ef2133ef9eed99a742a59ae644a',
 'frozen_before_test': True}

## 3 · Test 2023–24 and holdout 2025–26 (seen periods), weekly decisions

In [4]:
HOLD = config.ROOT / "data" / "holdout"
raw_h = pd.read_csv(HOLD / "spy_20261013.csv", index_col=0, parse_dates=True); px_h = raw_h[["Open", "Close"]]
rf_h = v2.load_tbill_daily(raw_h.index, HOLD / "irx_20261013.csv")
RES = {}
for period, data, prices, rfs, years in [("2023–24 test", ds, px, rf, [2023, 2024]),
                                         ("2025–26 holdout", build_dataset_v2(raw_h), px_h, rf_h, [2025, 2026])]:
    wf = v3.walk_forward_proba(data, KIND, years, PARAMS, MODEL)
    er = v3.expected_return(wf)
    held = v2.held_from_positions(weekly.weekly_positions_from_er(er, DEFAULT), start=1 if DEFAULT == "long" else 0)
    d = er.index; p_ = prices.loc[d]; r_ = rfs.loc[d]
    s = v2.summarize(v2.simulate(held, p_, r_), held, r_, f"v3w {PICK}")
    bhh = v2.buy_and_hold_held(d)
    b = v2.summarize(v2.simulate(bhh, p_, r_), bhh, r_, "Buy & Hold")
    lab = data.loc[d, "label"].dropna().astype(int)
    pr = wf.loc[lab.index, config.CLASSES]
    cls = classification_metrics(lab, np.array(config.CLASSES)[pr.to_numpy().argmax(1)], pr.to_numpy())
    pct = None
    if s["round_trips"] > 1:
        rand = v2.random_timing_benchmark(held, p_, r_, n=1000)
        pct = round(float((rand < s["final_value"]).mean()), 3)
    RES[period] = {"strategy": s, "buy_and_hold": b, "classification": {k: cls[k] for k in ["n", "accuracy", "macro_f1", "log_loss"]},
                   "beats_random": pct, "er_min": float(er.min()), "er_max": float(er.max())}
    display(pd.DataFrame([s, b]).set_index("strategy"))
    print(period, "| beats random:", pct if pct is not None else "not applicable (≤1 switch)", "| classification:", RES[period]["classification"])

def no_nan(o):
    if isinstance(o, dict): return {k: no_nan(v) for k, v in o.items()}
    if isinstance(o, list): return [no_nan(v) for v in o]
    return None if isinstance(o, float) and np.isnan(o) else o
(OUT / "results.json").write_text(json.dumps(no_nan(json.loads(json.dumps(
    {"frozen": FROZEN, "calibration": json.loads(calib.to_json(orient="index")), "development": json.loads(dev.to_json(orient="index")),
     "development_bh": bh, "results": RES}, default=str))), indent=2, allow_nan=False))
print("saved outputs/weekly/v3/")

,final_value,total_return,cagr,ann_vol,sharpe_excess,max_drawdown,round_trips,exposure,total_fees
strategy,,,,,,,,,
v3w C3-L,156546.25,0.5655,0.2523,0.1283,1.43,-0.0997,1,1.0,99.9
Buy & Hold,156546.25,0.5655,0.2523,0.1283,1.43,-0.0997,1,1.0,99.9


2023–24 test | beats random: not applicable (≤1 switch) | classification: {'n': 496, 'accuracy': 0.4819, 'macro_f1': 0.3388, 'log_loss': 1.0215}


,final_value,total_return,cagr,ann_vol,sharpe_excess,max_drawdown,round_trips,exposure,total_fees
strategy,,,,,,,,,
v3w C3-L,134409.74,0.3441,0.1823,0.1694,0.844,-0.1876,1,1.0,99.9
Buy & Hold,134409.74,0.3441,0.1823,0.1694,0.844,-0.1876,1,1.0,99.9


2025–26 holdout | beats random: not applicable (≤1 switch) | classification: {'n': 439, 'accuracy': 0.4989, 'macro_f1': 0.3619, 'log_loss': 1.0424}
saved outputs/weekly/v3/
